# opentargets-py — Demo Notebook

This notebook demonstrates the core features of the `opentargets-py` library against the live API.

**Topics covered:**
1. Installation & import
2. Target (gene/protein) queries
3. Disease queries
4. Drug queries
5. Search
6. Target → Disease associations
7. Batch queries
8. Pandas DataFrame output
9. Cache & performance

> API: https://api.platform.opentargets.org/api/v4/graphql

In [ ]:
# Install required packages (first run only)
# !pip install opentargets-py pandas

from opentargets import OpenTargetsClient

# Create client — cache is enabled by default (5 min TTL)
client = OpenTargetsClient()
print("✓ Client ready")

## 1. Target Queries

Query by gene symbol (`EGFR`) or Ensembl ID (`ENSG00000146648`).  
The library resolves symbols to Ensembl IDs automatically and caches the result.

In [ ]:
# Query by symbol
target = client.get_target("EGFR")

print(f"ID          : {target.id}")
print(f"Symbol      : {target.approved_symbol}")
print(f"Full name   : {target.approved_name}")
print(f"Biotype     : {target.biotype}")
if target.function_descriptions:
    print(f"Function    : {target.function_descriptions[0][:120]}...")

In [ ]:
# Query directly by Ensembl ID (skips symbol resolution, faster)
target2 = client.get_target("ENSG00000157764")  # BRAF
print(f"{target2.approved_symbol} — {target2.approved_name}")

## 2. Disease Queries

Fetch disease information by EFO identifier.

In [ ]:
disease = client.get_disease("EFO_0003060")  # lung carcinoma

print(f"ID              : {disease.id}")
print(f"Name            : {disease.name}")
print(f"Description     : {disease.description[:150]}...")
print(f"Therapeutic area: {', '.join(disease.therapeutic_areas[:3])}")

## 3. Drug Queries

In [ ]:
drug = client.get_drug("CHEMBL939")  # Erlotinib

print(f"ID               : {drug.id}")
print(f"Name             : {drug.name}")
print(f"Type             : {drug.drug_type}")
print(f"Mechanism        : {drug.mechanism_of_action}")
print(f"Clinical phase   : {drug.max_clinical_trial_phase}")
print(f"Trade names      : {', '.join(drug.trade_names)}")

In [ ]:
# Drug indications (approved diseases)
indications = client.get_drug_indications("CHEMBL939")
print(f"Erlotinib indications ({len(indications)} total):\n")
for ind in indications[:5]:
    print(f"  • {ind.disease_name:<45} phase={ind.max_phase_for_indication}")

## 4. Search

Search for targets, diseases, or drugs using free text.

In [ ]:
# Search across all entity types
results = client.search("BRCA1", limit=5)
print("Search results for 'BRCA1':\n")
for r in results:
    print(f"  [{r.entity_type:<8}] {r.name:<30} {r.id}")

In [ ]:
# Search diseases only
disease_results = client.search("breast cancer", entity_type="disease", limit=5)
print("Disease search results for 'breast cancer':\n")
for r in disease_results:
    print(f"  {r.name:<45} {r.id}")

## 5. Target → Disease Associations

Fetch which diseases a gene is associated with, along with evidence scores.

In [ ]:
associations = client.get_target_associations("EGFR", limit=10)

print(f"Top {len(associations)} diseases associated with EGFR:\n")
print(f"{'Disease':<45} {'Score':>6}  {'Datasources':>18}")
print("-" * 75)
for a in associations:
    n_ds = len([ds for ds in a.datasource_scores if ds.score > 0])
    print(f"{a.disease_name:<45} {a.score:>6.3f}  {n_ds:>18}")

In [ ]:
# Score for a specific target–disease pair
assoc = client.get_associations(
    target_id="ENSG00000146648",  # EGFR
    disease_id="EFO_0003060",     # lung carcinoma
)

if assoc:
    print(f"EGFR ↔ Lung carcinoma score: {assoc.score:.4f}")
    print("\nPer-datasource scores:")
    for ds in sorted(assoc.datasource_scores, key=lambda x: x.score, reverse=True):
        bar = "█" * int(ds.score * 20)
        print(f"  {ds.id:<35} {ds.score:.3f} {bar}")

## 6. Batch Queries

Fetch multiple genes in a single API call.

In [ ]:
import time

# One by one — 4 separate API calls
t0 = time.time()
targets_one_by_one = [client.get_target(g) for g in ["EGFR", "BRAF", "KRAS", "TP53"]]
t1 = time.time()

# Batch — 1 API call
targets_batch = client.get_targets(["EGFR", "BRAF", "KRAS", "TP53"])
t2 = time.time()

print(f"One by one: {t1-t0:.2f}s  (appears fast because results are cached)")
print(f"Batch     : {t2-t1:.2f}s\n")

print("Batch results:")
for t in targets_batch:
    print(f"  {t.approved_symbol:<8} {t.id}  —  {t.biotype}")

## 7. Pandas DataFrame Output

Pass `as_dataframe=True` to any list method to get a DataFrame directly.

In [ ]:
df = client.get_target_associations("EGFR", limit=50, as_dataframe=True)

print(f"DataFrame shape: {df.shape}\n")
print(df[["disease_name", "score", "evidence_count"]].head(10).to_string(index=False))

In [ ]:
# Filter and sort by score
top = df[df["score"] >= 0.5].sort_values("score", ascending=False)
print(f"Diseases with score ≥ 0.5: {len(top)}\n")
print(top[["disease_name", "score"]].head(10).to_string(index=False))

## 8. Error Handling

`NotFoundError` is raised for entities that cannot be found.

In [ ]:
from opentargets import NotFoundError, APIError

# Non-existent gene
try:
    client.get_target("NOSUCHGENE123")
except NotFoundError as e:
    print(f"NotFoundError: entity_type={e.entity_type!r}, entity_id={e.entity_id!r}")

# Non-existent disease
try:
    client.get_disease("EFO_9999999")
except NotFoundError as e:
    print(f"NotFoundError: {e}")

## 9. Cache Performance

The second call for the same query is served from cache — no API request is made.

In [ ]:
fresh_client = OpenTargetsClient(cache=True)

# First request — goes to API
t0 = time.time()
_ = fresh_client.get_target("EGFR")
t1 = time.time()
print(f"1st request (API)  : {(t1-t0)*1000:.0f} ms")

# Second request — served from cache
t2 = time.time()
_ = fresh_client.get_target("EGFR")
t3 = time.time()
print(f"2nd request (cache): {(t3-t2)*1000:.1f} ms")

# Cache disabled
no_cache = OpenTargetsClient(cache=False)
t4 = time.time()
_ = no_cache.get_target("EGFR")
t5 = time.time()
print(f"cache=False (always API): {(t5-t4)*1000:.0f} ms")

## 10. Clean Usage with Context Manager

HTTP connections are closed automatically when the `with` block exits.

In [ ]:
with OpenTargetsClient() as c:
    tp53 = c.get_target("TP53")
    print(f"{tp53.approved_symbol}: {tp53.approved_name}")

    drugs = c.get_target_drugs("TP53")
    print(f"Known drugs targeting TP53: {len(drugs)}")

print("✓ Connection closed")